# Public MPC catalogs and LSST restricted products 

> **Fallback edition.** Set `USE_FALLBACK = True` (default here) and **Run all** to use packaged CSV/PNG files when the live service is unavailable. Set it `False` to call the live services.


- **Tools:** [User Query / TAP](https://userquery.linea.org.br) and the [User Query guide](https://docs.linea.org.br/en/sci-platforms/user_query.html), [`pyvo`](https://pyvo.readthedocs.io/), [JupyterHub](https://jupyter.linea.org.br) (image **SolarSystem**), [Rubin DP1 documentation](https://dp1.lsst.io/).
- **Data products:**
  - **Public (everyone):** The MPC SBN mirror `mpc_sbn`, which includes observations submitted by Rubin tagged with `stn='X05'`. Once Rubin sends these observations to the MPC, they become open public data.
  - **Restricted (LSST data rights only):** The internal tables `lsst_dp1` / `lsst_dp2` (`ssobject`, `sssource`, `diasource`). The restricted track is covered in the last section of this tutorial. If you cannot see these tables, it is due to data-rights access policies, not a login error.
- **Authentication and authorization:** Logging into the LIneA website in your browser does **not** automatically attach your credentials to `pyvo` in this notebook. The public catalogues allow an anonymous TAP session. However, the private catalogues require two steps: **authentication** (an API token in the `Authorization` header to prove who you are) and **authorization** (the system verifying if your account has LSST data rights). Even if you have data rights, sending an empty `requests.Session()` makes you anonymous, and User Query will reject access to `lsst_dp1` / `lsst_dp2`. You must get your token in [User Query](https://userquery.linea.org.br): click your username (top right) → **API token**, as described in the [User Query guide](https://docs.linea.org.br/en/sci-platforms/user_query.html). The first code cell sets `API_TOKEN = "Token YOUR_TOKEN_HERE"`. Please replace that placeholder with your actual token before starting section 4.
- **Prerequisites:** A LIneA account ([register here](https://register.linea.org.br)), plus basic knowledge of Python, Jupyter, and Solar System photometry. The restricted track also requires the API token and LSST data rights on that same account.
- **Credit:** LIneA User Query documentation, RTN-045, DOI [10.11578/rubin/dc.20250909.20](https://doi.org/10.11578/rubin/dc.20250909.20).
- **Get Support:** LIneA Science Platform forum (`help`) and [data.linea.org.br](https://data.linea.org.br).

TAP and HiPS-style services take your query directly to the database and only return the filtered subset of data. Please do not attempt to download the entire `mpc_sbn.obs_sbn` table. Every query below includes a selective filter and a `LIMIT` to avoid overloading the system.

### Agenda

| Section | Tool |
| --- | --- |
| 1. Getting ready and TAP | User Query / `pyvo` |
| 2. Public MPC and `stn='X05'` | TAP, PostgreSQL |
| 3. Photometric light curves | TAP + matplotlib |
| 4. Restricted DP1/DP2 | TAP, data rights |
| 5. Porting SDM ↔ LIneA names (optional) | Documentation |

The first code cell has `USE_FALLBACK`. Leave it `True` (default in this edition) for packaged files. Set it `False` and **Run all** to load packaged CSV files instead. There is no packaged fallback for `lsst_dp1`.


## 1. Getting ready and TAP

| Track | What | Access |
| --- | --- | --- |
| **A — Public** | `mpc_sbn` on TAP, including `stn='X05'` | Everyone |
| **B — Restricted** | `lsst_dp1` / `lsst_dp2` | LSST data rights only (section 4) |

Open [User Query](https://userquery.linea.org.br) in your browser and confirm you can see the metadata pages. The published schemas are listed at [data.linea.org.br/database](https://data.linea.org.br/en/database/index.html): [des_dr2](https://userquery.linea.org.br/metadata/des_dr2/), [des_y6_gold](https://userquery.linea.org.br/metadata/des_y6_gold/), [gaia_dr3](https://userquery.linea.org.br/metadata/gaia_dr3/), [mpc_sbn](https://userquery.linea.org.br/metadata/mpc_sbn/), and [twomass](https://userquery.linea.org.br/metadata/twomass/) are public. [lsst_dp1](https://userquery.linea.org.br/metadata/lsst_dp1/) and [lsst_dp2](https://userquery.linea.org.br/metadata/lsst_dp2/) are restricted.

### What TAP and `pyvo` are

The IVOA Table Access Protocol (TAP) allows you to send a query string to a remote astronomical database and returns a VOTable. You do not need to open a complex database tunnel for the public catalogues; the TAP service acts as the front door.

```python
API_TOKEN = "Token YOUR_TOKEN_HERE"
session = requests.Session()
session.headers["Authorization"] = API_TOKEN
tap = pyvo.dal.TAPService("https://userquery.linea.org.br/tap", session=session)
```

This Python client connects this notebook to the LIneA User Query service, as detailed in the [User Query guide](https://docs.linea.org.br/en/sci-platforms/user_query.html).

A session without a token is considered anonymous. This is the correct way to access the public `mpc_sbn` tables. However, this same client will only be able to read `lsst_dp1` and `lsst_dp2` after the `Authorization` header is properly set with your API token. The token provides authentication (it identifies who you are). User Query then checks authorization (whether your identified account has LSST data rights). Remember that being logged into the website in your browser does not automatically send your login to `pyvo`.

To test this, copy the token from User Query → username → **API token** and paste it into `API_TOKEN` in the next cell (make sure to include the word `Token` and a space), then run the cell again. As long as `API_TOKEN` contains the placeholder `Token YOUR_TOKEN_HERE`, the header remains empty and the cell will print `anonymous`. For your security, the code will never print your actual token value on the screen.

Confirm that the next cell prints the TAP base URL and either `TAP session: anonymous` or `TAP session: authenticated`.

| Mode | Call | When |
| --- | --- | --- |
| Synchronous | `tap.run_sync(query, language="postgresql")` | Used in these cells; results arrive in seconds. |
| Asynchronous | `job = tap.submit_job(query, language="postgresql", QUEUE="five_minutes")` then `job.run()` | Used for heavy scans. Queues include `default` (~30 s), `five_minutes`, and `two_hours`. |

This notebook wraps synchronous calls in `run_tap_df(tap, query)`. LIneA VOTables currently mark timestamp columns with `xtype="timestamp"` but omit `arraysize`, so a plain `.to_table().to_pandas()` truncates values such as `2026-10-23 ...` to `2`. The helper patches those FIELD tags before parsing.

The exact same endpoint accepts two different SQL dialects:

|  | ADQL (default) | PostgreSQL (`language="postgresql"`) |
| --- | --- | --- |
| Row cap | `SELECT TOP 5 ...` | `SELECT ... LIMIT 5` |
| Conditional aggregates | Limited | `COUNT(*) FILTER (WHERE ...)` |
| Spatial search | `CONTAINS(POINT('ICRS', ra, dec), CIRCLE(...))=1` | Indexed `q3c_radial_query(ra, dec, ra0, dec0, r)` |

This notebook uses PostgreSQL. It is highly recommended to inspect the column names in the metadata UI before writing your own query. Always remember: never use `SELECT *` from `obs_sbn` without adding `WHERE` (to filter) and `LIMIT` (to cap the rows).

`stn = 'X05'` is the official MPC observatory code for the Vera C. Rubin Observatory. Running a count query directly on the server avoids downloading the entire massive table. For context, a count measured on October 20, 2026, was `n_rubin = 8084389`. The next section will download only a 100-row window, not the full dataset.

### Reference join (not executed in section 2)

In SQL, `SELECT` chooses columns, `FROM` specifies the table, `WHERE` filters the rows, `ORDER BY` sorts them, and `LIMIT` bounds the final result size (ADQL would use `TOP` instead of `LIMIT`). Section 2 below does not run this heavy join. Instead, it runs three bounded, lightweight queries: a 100-row Rubin window, a single orbit, and the specific observations for `2026 ED138`.

```sql
SELECT o.provid, o.obstime, o.mag, o.band, m.a, m.e, m.i
FROM mpc_sbn.obs_sbn AS o
JOIN mpc_sbn.mpc_orbits AS m
  ON m.unpacked_primary_provisional_designation = o.provid
WHERE o.stn = 'X05' AND o.deprecated IS NULL
ORDER BY o.obstime
LIMIT 20
```

In [ ]:
%matplotlib inline

import re
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvo
import requests
from astropy.io.votable import parse as parse_votable

# True: skip live calls and use the fallback cell under each section.
USE_FALLBACK = True

# User Query → username → API token. Form from the User Query guide: "Token <value>".
# Leave the placeholder for the public mpc_sbn track. Replace it before section 4.
# Pasting only the hex value is fine: the cell adds the "Token " prefix.
API_TOKEN = "Token YOUR_TOKEN_HERE"

BAND_STYLE = {
    "V": ("#440154", "o"),
    "R": ("#31688e", "s"),
    "G": ("#35b779", "^"),
    "r": ("#fde725", "D"),
    "g": ("#21918c", "v"),
    "i": ("#5ec962", "P"),
}


def run_tap_df(tap, query, language="postgresql"):
    """Run a TAP query and return a pandas DataFrame.

    LIneA VOTables mark timestamps with xtype="timestamp" but omit arraysize,
    so a plain ``tap.run_sync(...).to_table().to_pandas()`` truncates values
    like ``2026-10-23 ...`` to ``2``. Patch those FIELD tags before parsing.
    """
    raw = tap.create_query(query, language=language).execute_stream().read()
    text = raw.decode("utf-8")
    text = re.sub(
        r'(<FIELD\b(?=[^>]*\bxtype="timestamp")[^>]*?)(/?>)',
        r'\1 datatype="char" arraysize="*"\2',
        text,
    )
    votable = parse_votable(BytesIO(text.encode("utf-8")), verify="ignore")
    resource = votable.resources[0]
    for info in resource.infos:
        if info.name == "QUERY_STATUS" and str(info.value).lower() not in ("ok", "overflow"):
            raise RuntimeError(f"TAP query failed: {info.value} {info.content}")
    return resource.tables[0].to_table().to_pandas()


def plot_ed138_sky(obs):
    """Sky track of the packaged or live 2026 ED138 sample. RA increases to the left."""
    fig_sky, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
    ax.plot(obs["ra"], obs["dec"], color="0.45", lw=0.8, zorder=1)
    ax.scatter(obs["ra"], obs["dec"], s=22, c="0.15", marker="o", zorder=2)
    ax.invert_xaxis()
    ax.set_xlabel("Right ascension / deg (east to the left)")
    ax.set_ylabel("Declination / deg")
    ax.set_title("2026 ED138: observations in this sample")
    plt.close(fig_sky)
    return fig_sky


def plot_mpc_lightcurve(phot, title, outfile):
    """Plot apparent magnitude versus time. Marker and color encode the band."""
    frame = phot.copy()
    frame["obstime"] = pd.to_datetime(frame["obstime"], utc=True, format="mixed")
    frame["mag"] = pd.to_numeric(frame["mag"], errors="coerce")
    frame = frame.dropna(subset=["obstime", "mag"]).sort_values("obstime")
    if "band" not in frame.columns:
        frame["band"] = "?"
    else:
        frame["band"] = frame["band"].astype(str)

    fig_dir = Path("data/figures")
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(9.5, 4.2), constrained_layout=True)
    for band_name, group in frame.groupby("band", sort=True):
        color, marker = BAND_STYLE.get(band_name, ("0.4", "x"))
        ax.scatter(
            group["obstime"],
            group["mag"],
            s=22,
            alpha=0.85,
            c=color,
            marker=marker,
            label=str(band_name),
            edgecolors="none",
        )
    ax.invert_yaxis()
    ax.set_xlabel("Observation time (UTC)")
    ax.set_ylabel("Apparent magnitude (brighter up)")
    ax.set_title(title)
    ax.legend(title="band", loc="best", fontsize=8)
    fig.savefig(fig_dir / outfile, dpi=150)
    plt.close(fig)
    return fig


if USE_FALLBACK:
    print("Skipped: USE_FALLBACK is True.")
    tap = None
    session = None
else:
    session = requests.Session()
    if API_TOKEN and "YOUR_TOKEN_HERE" not in API_TOKEN:
        token_str = API_TOKEN.strip()
        if not token_str.lower().startswith("token "):
            token_str = f"Token {token_str}"
        session.headers["Authorization"] = token_str
        print("TAP session: authenticated. The token value is not printed.")
    else:
        print("TAP session: anonymous. mpc_sbn works.")
        print("Replace API_TOKEN above to read lsst_dp1 / lsst_dp2.")

    tap = pyvo.dal.TAPService("https://userquery.linea.org.br/tap", session=session)
    print(tap.baseurl)


In [ ]:
if USE_FALLBACK:
    print("USE_FALLBACK is True. Later cells read the packaged CSV files.")
else:
    print(tap.baseurl)
    authenticated = "Authorization" in session.headers
    print("TAP session:", "authenticated" if authenticated else "anonymous")

## 2. Public MPC catalogs and Rubin at the MPC

This section relies exclusively on TAP. It does not call the Solar System Portal directly.

The Rubin Observatory submits its observations to the IAU Minor Planet Center (MPC). That data stream is public. LIneA mirrors this public data as **`mpc_sbn`**. Filtering by `stn = 'X05'` selects the Rubin contribution **without** requiring LSST data rights, because these rows are copies of the open MPC data, not the internal `lsst_dp1` or `lsst_dp2` releases.

| Table | Role | Columns used below |
| --- | --- | --- |
| `mpc_sbn.mpc_orbits` | Orbits and designations | `unpacked_primary_provisional_designation`, `a`, `e`, `i`, `q`, `h`, `nobs_total`, `arc_length_total`, `orbit_type_int` |
| `mpc_sbn.obs_sbn` | ADES observations | `permid`, `provid`, `stn`, `obstime`, `ra`, `dec`, `mag`, `rmsmag`, `band` |

In the orbit query below, we rename some columns (like `h AS "H"` and `orbit_type_int AS orbit_type`) so that the live column names match the standard magnitude and class labels you are used to. Adding `deprecated IS NULL` removes withdrawn submissions. The `status` values `P`, `p`, and `I` mark published, provisional, and isolated tracklets, but our sample does not filter on `status`.

Our target example for this tutorial is the TNO candidate **2026 ED138**. The Rubin window used here is from April 21 to May 5, 2025, capped at `LIMIT 100` and ordered by `obstime` so that the sample is reproducible.

Confirm that the output gives you one orbit row, a non-empty `2026 ED138` observation table, and exactly 100 Rubin rows (it will be fewer only if the chosen time window itself is shorter).

**Exercise 1 (MPC / TAP).** Change the designation to another `provid`, keep `deprecated IS NULL`, and run the station summary. Report the number of distinct stations (`len(ex1)`) and the total observations (`ex1["n"].sum()`). `LIMIT 100` caps the number of station rows after `GROUP BY`, not the raw observation count.

> *Hint:* Do not remove `LIMIT`. Confirm you have at most 100 station rows. You can read candidate designations from the `provid` column of the live `x05` table. Note that the packaged `mpc_obs_sample_head.csv` has no designations, and `2026 ED138` is the worked example we already queried above.

In [ ]:
if USE_FALLBACK:
    print("Skipped: USE_FALLBACK is True.")
else:
    # Rubin (X05) observations in a fixed 2025 window, capped at 100 rows.
    sql_x05 = """
    SELECT obstime, ra, dec, mag, rmsmag, band, stn, provid, rmsra, rmsdec
    FROM mpc_sbn.obs_sbn
    WHERE stn = 'X05' AND deprecated IS NULL
      AND obstime >= '2025-04-21'
      AND obstime < '2025-05-05'
    ORDER BY obstime
    LIMIT 100
    """
    # Orbital elements for the tutorial target 2026 ED138.
    sql_orbit = """
    SELECT unpacked_primary_provisional_designation, a, e, i, q,
           h AS "H", nobs_total, arc_length_total,
           orbit_type_int AS orbit_type
    FROM mpc_sbn.mpc_orbits
    WHERE unpacked_primary_provisional_designation = '2026 ED138'
    LIMIT 1
    """
    # All non-deprecated MPC observations of 2026 ED138 (any station).
    sql_obs = """
    SELECT obstime, ra, dec, mag, rmsmag, band, stn, provid, rmsra, rmsdec
    FROM mpc_sbn.obs_sbn
    WHERE provid = '2026 ED138' AND deprecated IS NULL
    ORDER BY obstime
    LIMIT 500
    """

    x05 = run_tap_df(tap, sql_x05)
    orbit = run_tap_df(tap, sql_orbit)
    obs = run_tap_df(tap, sql_obs)
    print("X05 rows:", len(x05), "| ED138 orbit rows:", len(orbit), "| ED138 observations:", len(obs))
    display(orbit)
    display(obs.head())
    fig_sky = plot_ed138_sky(obs)
    fig_sky

In [ ]:
import utils

if USE_FALLBACK:
    x05 = utils.load_fallback_csv("mpc_obs_sample_head.csv")
    orbit = utils.load_fallback_csv("mpc_2026ED138_orbit.csv")
    obs = utils.load_fallback_csv("mpc_2026ED138_obs.csv")
    obs_cols = [
        c for c in ["obstime", "ra", "dec", "mag", "rmsmag", "band", "stn", "provid", "rmsra", "rmsdec"]
        if c in obs.columns
    ]
    print("Fallback CSV | X05 rows:", len(x05), "| ED138 orbit rows:", len(orbit), "| ED138 observations:", len(obs))
    display(orbit)
    display(obs[obs_cols].head())
    fig_sky = plot_ed138_sky(obs)
    fig_sky
else:
    print(
        "Live TAP tables kept:",
        len(x05), "X05 rows |",
        len(orbit), "ED138 orbit rows |",
        len(obs), "ED138 observations.",
    )

> **Figure 1:** Sky positions of `2026 ED138` in this sample. The line connects points in `obstime` order. Right ascension increases to the left.
>
> *Alt-text: Declination versus right ascension. Points mark individual observations. The horizontal axis is reversed so east is to the left.*

In [ ]:
# Exercise 1. Change provid, keep the filters, and read the station counts.
provid = "2026 ED138"

# Observation counts per station for one object.
sql_ex1 = f"""
SELECT stn, COUNT(*) AS n
FROM mpc_sbn.obs_sbn
WHERE provid = '{provid}' AND deprecated IS NULL
GROUP BY stn
ORDER BY n DESC
LIMIT 100
"""
if USE_FALLBACK:
    print("Skipped: USE_FALLBACK is True.")
else:
    ex1 = run_tap_df(tap, sql_ex1)
    print("stations:", len(ex1), "| total obs:", int(ex1["n"].sum()))
    display(ex1)

## 3. MPC photometry as a light curve

We will target **(4) Vesta** (`permid = '4'`), and then **(433) Eros** (`permid = '433'`). This section focuses only on **downloading** the magnitudes and plotting them. Advanced steps like period search, phase folding, and color corrections are left for your own scientific analysis later.

Each MPC magnitude represents disk-integrated light at a specific geometry and a specific photometric band. Rotation changes the visible cross-section of the object on hour timescales. Furthermore, changing geometry (heliocentric distance, geocentric distance, phase angle) alters the baseline magnitude across nights and different oppositions. Using `ORDER BY obstime DESC LIMIT 500` without filtering specific dates might mix observations from several different oppositions. This is intentional for this workshop: the goal here is to learn how to extract the data table, not how to perform the final scientific data reduction.

Using `mag IS NOT NULL` drops any rows that do not have a recorded brightness. The uncertainty column is named `rmsmag` in the live ADES table, though it might be named `magerr` in the packaged fallback file.

A previous offline run of the packaged Vesta file returned **480** rows, and the packaged Eros file returned **420** rows. A live query using `ORDER BY obstime DESC LIMIT 500` might return different results because it fetches the 500 most recent magnitudes, rather than what is frozen in the file. The plotting function automatically re-sorts them in ascending order for display.

> **Accessibility note:** Do not try to read the band from the color alone. Each photometric band is also represented by its own specific marker shape.

Check the results to confirm that each table contains rows and that the light-curve figure displays points spanning across the time window.

The subsequent cells load Eros using the same methodology. Period analysis is not included in this notebook.

**Exercise 2 (light curve).** After generating the Eros figure, modify the code to keep only a single band. Change the `band_only` variable to one of `V`, `R`, `G`, or `r`, and report the final row count for that specific band.

> *Hint:* These four bands are guaranteed to be present on the packaged file. Confirm that the printed row count is smaller than the full table's row count.

In [ ]:
if USE_FALLBACK:
    print("Skipped: USE_FALLBACK is True.")
else:
    # Photometry for (4) Vesta: rows with magnitude, newest first.
    sql_vesta = """
    SELECT permid, provid, obstime, ra, dec, mag, rmsmag, band, stn, status
    FROM mpc_sbn.obs_sbn
    WHERE permid = '4' AND mag IS NOT NULL
    ORDER BY obstime DESC
    LIMIT 500
    """
    phot = run_tap_df(tap, sql_vesta)
    print("Vesta rows:", len(phot))
    display(phot.head())
    fig_vesta = plot_mpc_lightcurve(phot, "(4) Vesta: MPC light curve", "vesta_lightcurve.png")
    fig_vesta

In [ ]:
import utils

if USE_FALLBACK:
    phot = utils.load_fallback_photometry("vesta")
    print("Fallback photometry | Vesta rows:", len(phot))
    display(phot.head())
    fig_vesta = plot_mpc_lightcurve(phot, "(4) Vesta: MPC light curve", "vesta_lightcurve.png")
    fig_vesta
else:
    print(f"Live Vesta table kept: {len(phot)} rows.")

> **Figure 2:** Apparent magnitude versus time for (4) Vesta. Marker shape and color both encode the photometric band.
>
> *Alt-text: Magnitude versus UTC time. Brighter points are higher. Circles, squares, and triangles mark different bands.*

In [ ]:
if USE_FALLBACK:
    print("Skipped: USE_FALLBACK is True.")
else:
    # Photometry for (433) Eros: rows with magnitude, newest first.
    sql_eros = """
    SELECT permid, provid, obstime, ra, dec, mag, rmsmag, band, stn, status
    FROM mpc_sbn.obs_sbn
    WHERE permid = '433' AND mag IS NOT NULL
    ORDER BY obstime DESC
    LIMIT 500
    """
    eros = run_tap_df(tap, sql_eros)
    print("Eros rows:", len(eros))
    display(eros.head())
    fig_eros = plot_mpc_lightcurve(eros, "(433) Eros: MPC light curve", "eros_lightcurve.png")
    fig_eros

In [ ]:
import utils

if USE_FALLBACK:
    eros = utils.load_fallback_photometry("eros")
    print("Fallback photometry | Eros rows:", len(eros))
    display(eros.head())
    fig_eros = plot_mpc_lightcurve(eros, "(433) Eros: MPC light curve", "eros_lightcurve.png")
    fig_eros
else:
    print(f"Live Eros table kept: {len(eros)} rows.")

> **Figure 3:** The same style of light curve for (433) Eros.
>
> *Alt-text: Magnitude versus UTC time for Eros, with bands marked by shape and color.*

In [ ]:
# Exercise 2. Change band_only to V, R, G, or r and report the row count.
band_only = "V"

one = eros.loc[eros["band"].astype(str) == band_only].copy()
print(band_only, "rows:", len(one))
if len(one) < 5:
    print("Too few rows in that band. Pick another value of band_only.")
else:
    display(one.head())
    fig_band = plot_mpc_lightcurve(
        one,
        f"(433) Eros: band {band_only} only",
        f"eros_lightcurve_{band_only}.png",
    )
    fig_band

## 4. Wrap-up and the restricted track

### Public path

1. Execute a TAP query on `mpc_sbn`, using a specific filter and a `LIMIT`.
2. Extract Rubin rows as open MPC observations using `stn = 'X05'`.
3. Extract one orbit and its corresponding observations for `2026 ED138`.
4. Generate photometric light curves for Vesta and Eros based on `mpc_sbn.obs_sbn`.

Please note that occultation planning is covered entirely in `tutorial_occultations.ipynb`. That notebook uses a web-based HTTP API, which is different from the SQL approach taught here.

### What the restricted tables add

You can stop reading here if you do not have LSST data rights. If you cannot query `lsst_dp*` entries, it is simply because of data access policies, not a login failure or a system bug. If you *do* have data rights, go back to the first code cell, replace `API_TOKEN` with your actual token, and re-run section 1 until it prints `TAP session: authenticated`. Being logged into the browser does not authenticate `pyvo`. Please do not confuse `stn = 'X05'` (which is open public MPC data) with the SSP tables (which are strictly restricted).

| | Public `mpc_sbn.obs_sbn` | Restricted SSP (`sssource` / `ssobject`) |
| --- | --- | --- |
| Epoch and sky | `obstime`, `ra`, `dec` | `midpoint_mjd_tai`, ephemeris, residuals |
| Photometry | Heterogeneous MPC `mag` / `band` | LSST `ugrizy` |
| Geometry | Not in the MPC row | `helio_range`, `topo_range`, `phase_angle`, `elongation`, `eph_rate` |
| Phase curve | External | Per-band `H` / `G12` parameters on `SSObject` |

Science documentation: [dp1.lsst.io](https://dp1.lsst.io/products/catalogs/index.html), [dp2.lsst.io](https://dp2.lsst.io/), [sdm-schemas.lsst.io](https://sdm-schemas.lsst.io/). The LIneA names for the same products are listed in `vera_rubin/lsst_dp1/lsst_dp1.yml` and `vera_rubin/lsst_dp2/lsst_dp2.yml`. Section 5 at the end of this notebook explains how those names map to the official SDM spelling.

When joining `sssource` to `diasource` on `dia_source_id` at LIneA, filter `ss_object_id` on **both** sides of the join so the database can use the index:

```sql
SELECT sss.ss_object_id, sss.helio_range, sss.topo_range, sss.phase_angle,
       dia.midpoint_mjd_tai, dia.band
FROM lsst_dp2.sssource AS sss
JOIN lsst_dp2.diasource AS dia
  ON sss.dia_source_id = dia.dia_source_id
WHERE sss.ss_object_id = 21163615737558602
  AND dia.ss_object_id = 21163615737558602
ORDER BY dia.midpoint_mjd_tai
LIMIT 100
```

The next cell attempts to read five `lsst_dp1.ssobject` rows using the session you created in section 1. The LIneA DP1 column is `num_obs`. If that earlier cell printed `TAP session: anonymous`, this query will not even be sent: since `API_TOKEN` is still the placeholder, the call is anonymous, and User Query will return a permission error even if your account actually possesses LSST data rights.

To fix this, replace `API_TOKEN` in the first code cell and re-run section 1 until it says `authenticated`. The token authenticates the call, and your data rights authorize the database read. Both steps are absolutely required. If your session is properly authenticated and the service still denies the query, it means your account does not have LSST data rights. That denial is the expected, correct outcome on the public track. If that happens, just continue your work with `mpc_sbn`. Note that there is no packaged fallback substitute for `lsst_dp1`.

**Exercise 3 (documentation only).** Using [DP1 MPCORB](https://dp1.lsst.io/products/catalogs/mpcorb.html) or [sdm-schemas](https://sdm-schemas.lsst.io/), name one column that the public `mpc_sbn.mpc_orbits` does not provide, and briefly explain why it is important for reduced magnitudes or phase curves.

> *Hint:* Look for `H` / `G12` parameters or a linkage field. The answer only needs to be one column name and a single explanatory sentence.

In [ ]:
if USE_FALLBACK:
    print("Skipped: USE_FALLBACK is True.")
else:
    # Restricted DP1 sample: first five ssobject rows (needs API token).
    sql_sso = """
    SELECT ss_object_id, num_obs
    FROM lsst_dp1.ssobject
    ORDER BY ss_object_id
    LIMIT 5
    """
    if "Authorization" not in session.headers:
        print("Not queried: this session has no Authorization header, so the call would be anonymous.")
        print("Replace API_TOKEN in the first code cell (User Query → username → API token) and re-run section 1.")
    else:
        sso = run_tap_df(tap, sql_sso)
        print("lsst_dp1.ssobject rows:", len(sso))
        display(sso)

In [ ]:
# There is no packaged lsst_dp1 table.
if USE_FALLBACK:
    print("Restricted track skipped (USE_FALLBACK). Continue with the public mpc_sbn / X05 path.")
elif "Authorization" not in session.headers:
    print("Restricted query was not sent: the TAP session is still anonymous.")
    print("Authentication is API_TOKEN in the first code cell. Authorization is LSST data rights. Both are required.")
else:
    print(f"Live lsst_dp1.ssobject table kept ({len(sso)} rows).")

### Restricted light curve (not executed)

The public MPC path plots heterogeneous `mag` / `band` from `obs_sbn`. On the restricted SSP path, time and filter live on `lsst_dp2.diasource` (`midpoint_mjd_tai`, `band`), and photometry is stored as flux in nJy — there is no magnitude column in the LIneA DP2 schema (`lsst_dp2.yml`). Prefer `science_flux` (forced PSF on the visit image) over `psf_flux` (difference-image flux). Convert to AB magnitude in SQL with `-2.5 * log(flux_nJy * 1.0) + 31.4` (PostgreSQL `log` is base-10; multiply by `1.0` so `real` flux promotes cleanly — `log(10, …)` is rejected here). Filter `band` to pull several LSST filters in one query, and keep `science_flux > 0` so the log is defined:

```sql
SELECT dia.midpoint_mjd_tai, dia.band,
       dia.science_flux, dia.science_flux_err,
       -2.5 * log(dia.science_flux * 1.0) + 31.4 AS science_mag,
       sss.phase_angle, sss.helio_range, sss.topo_range
FROM lsst_dp2.sssource AS sss
JOIN lsst_dp2.diasource AS dia
  ON sss.dia_source_id = dia.dia_source_id
WHERE sss.ss_object_id = 21163615737558602
  AND dia.ss_object_id = 21163615737558602
  AND dia.band IN ('g', 'r', 'i')
  AND dia.science_flux > 0
ORDER BY dia.midpoint_mjd_tai
LIMIT 500
```

This join is a DP2 example (`diasource` + `sssource`). Replace the `ss_object_id` with one from the `ssobject` sample above if needed. Plot `science_mag` (or `science_flux`) versus `midpoint_mjd_tai`, coloring by `band`, the same way section 3 colors MPC light curves.

## References

- LIneA User Query / TAP and `pyvo` — [docs.linea.org.br](https://docs.linea.org.br/en/sci-platforms/user_query.html)
- Schema metadata — [data.linea.org.br/database](https://data.linea.org.br/en/database/index.html): [des_dr2](https://userquery.linea.org.br/metadata/des_dr2/), [des_y6_gold](https://userquery.linea.org.br/metadata/des_y6_gold/), [gaia_dr3](https://userquery.linea.org.br/metadata/gaia_dr3/), [mpc_sbn](https://userquery.linea.org.br/metadata/mpc_sbn/), [twomass](https://userquery.linea.org.br/metadata/twomass/), [lsst_dp1](https://userquery.linea.org.br/metadata/lsst_dp1/) (restricted), [lsst_dp2](https://userquery.linea.org.br/metadata/lsst_dp2/) (restricted)
- DP1 / DP2 / SDM — [dp1.lsst.io](https://dp1.lsst.io/) · [dp2.lsst.io](https://dp2.lsst.io/) · [sdm-schemas.lsst.io](https://sdm-schemas.lsst.io/)
- RTN-045 · DOI [10.11578/rubin/dc.20250909.20](https://doi.org/10.11578/rubin/dc.20250909.20)

## 5. Porting column names: Rubin SDM / Qserv ↔ LIneA

This section is optional reading after the hands-on path. It does not replace section 4.

The Rubin Science Data Model and the LSST Qserv / RSP TAP services use **PascalCase table names** and **camelCase column names** (for example `SSSource.helioRange`). At LIneA, the same logical tables are exposed as **lowercase table names** and **snake_case column names** (for example `sssource.helio_range`). The meaning of each column is the same; only the spelling changes.

Official SDM browser: [sdm-schemas.lsst.io](https://sdm-schemas.lsst.io/) ([DP2 SSObject](https://sdm-schemas.lsst.io/dp2.html#SSObject), [DP2 SSSource](https://sdm-schemas.lsst.io/dp2.html#SSSource)). LIneA published names: `vera_rubin/lsst_dp1/lsst_dp1.yml` and `vera_rubin/lsst_dp2/lsst_dp2.yml`.

### Conversion rule

1. Table: `SSObject` → `ssobject`, `SSSource` → `sssource`, `DiaSource` → `diasource`.
2. Column: insert an underscore before each capital letter, then lower-case everything. Examples: `ssObjectId` → `ss_object_id`, `helioRange` → `helio_range`, `midpointMjdTai` → `midpoint_mjd_tai`.
3. Schema / catalog: on the Rubin side you often query a DP schema through Qserv; at LIneA you use `lsst_dp1` or `lsst_dp2` in User Query.
4. Dialect: many RSP examples use ADQL (`SELECT TOP n ...`). This notebook uses PostgreSQL (`LIMIT n`). Change only the row-cap keyword when you port the syntax.

| Concept | Rubin SDM / Qserv ([sdm-schemas.lsst.io](https://sdm-schemas.lsst.io/dp2.html)) | LIneA User Query (`lsst_dp2.yml`) |
| --- | --- | --- |
| SSO object table | `SSObject` | `lsst_dp2.ssobject` |
| SSO source table | `SSSource` | `lsst_dp2.sssource` |
| Difference-image source | `DiaSource` | `lsst_dp2.diasource` |
| Object id | `ssObjectId` | `ss_object_id` |
| Source id | `diaSourceId` | `dia_source_id` |
| Observation epoch | `midpointMjdTai` | `midpoint_mjd_tai` |
| Heliocentric distance | `helioRange` | `helio_range` |
| Topocentric distance | `topoRange` | `topo_range` |
| Phase angle | `phaseAngle` | `phase_angle` |
| Observation count (DP2) | `nObs` | `n_obs` |
| Observation count (DP1 at LIneA) | `numObs` (older SDM spelling) | `num_obs` in `lsst_dp1.ssobject` |

### Same science query on both sides

The join below asks for geometry from `SSSource` and the epoch/band from `DiaSource` for one Solar System object. Filter `ssObjectId` / `ss_object_id` on **both** sides of the join so the database can use the index.

Rubin SDM / Qserv style (camelCase, ADQL):

```sql
SELECT sss.ssObjectId, sss.helioRange, sss.topoRange, sss.phaseAngle,
       dia.midpointMjdTai, dia.band
FROM SSSource AS sss
JOIN DiaSource AS dia
  ON sss.diaSourceId = dia.diaSourceId
WHERE sss.ssObjectId = 21163615737558602
  AND dia.ssObjectId = 21163615737558602
ORDER BY dia.midpointMjdTai
```

LIneA User Query style (snake_case, PostgreSQL), matching `vera_rubin/lsst_dp2/lsst_dp2.yml`:

```sql
SELECT sss.ss_object_id, sss.helio_range, sss.topo_range, sss.phase_angle,
       dia.midpoint_mjd_tai, dia.band
FROM lsst_dp2.sssource AS sss
JOIN lsst_dp2.diasource AS dia
  ON sss.dia_source_id = dia.dia_source_id
WHERE sss.ss_object_id = 21163615737558602
  AND dia.ss_object_id = 21163615737558602
ORDER BY dia.midpoint_mjd_tai
LIMIT 100
```

To go the other way (LIneA → Qserv), reverse the same mapping: remove underscores and restore camelCase / PascalCase, and switch `LIMIT` to `TOP` if the client defaults to ADQL.

**Exercise 4 (name mapping).** Open [DP2 SSSource](https://sdm-schemas.lsst.io/dp2.html#SSSource) and the LIneA file `vera_rubin/lsst_dp2/lsst_dp2.yml`. Pick one geometry column (for example heliocentric distance). Write its SDM name and its LIneA name, then say in one sentence why that column matters for reduced magnitudes or phase curves.

> *Hint:* SDM `helioRange` maps to LIneA `helio_range`. You may also look at `H` / `G12` on [SSObject](https://sdm-schemas.lsst.io/dp2.html#SSObject).